# Playing with Cleaned and Parsed Data

# Key Libraries & Essential Functions
1. Pandas (Data Manipulation & Exploration)
Pandas is your primary tool for loading the JSON and analyzing tabular data.

**json.load() / pd.read_json()**: Load your extracted JSON file into memory.

**pd.json_normalize(df['parsed_data'])**: Essential for flattening nested JSON objects into dedicated DataFrame columns.

**df.head() / df.tail()**: Quick visual check of top/bottom rows.

**df.info() & df.describe()**: Check data types, null counts, and summary statistics (e.g., mean rug dimensions, average confidence scores).

**df.isnull().sum()**: Identify missing attributes across your parsed dataset.

**df['regional_style'].value_counts()**: Instantly count frequencies of categorical variables (e.g., most common regional styles or origin countries).

**df.groupby('is_sold')**: Compare metrics across categories (e.g., average sale price of sold vs. available rugs).

# Cleaning & Data Conversion Functions
**pd.to_numeric()**: Convert string prices (e.g., "$29,000.00") into float values for math operations.

**df.explode('materials')**: Unroll list values within columns (like materials: ["Wool", "Cotton"]) into individual rows to easily plot material distributions.

# Matplotlib & Seaborn (Data Visualization)
Visually spot outliers, dimension distributions, and pricing trends before building frontend UI components.

**sns.histplot(df['width_ft'])**: Inspect size distributions to inform size-filter ranges on your web app.

**sns.scatterplot(x='width_ft', y='length_ft', hue='is_sold', data=df)**: Plot rug aspect ratios and availability.

**sns.boxplot(x='regional_style', y='sale_price_cleaned', data=df)**: Analyze price ranges across styles.

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Set visual style
sns.set_theme(style="whitegrid")

# 1. Load raw JSON data
with open('generated_data/rugs_with_image_analysis.json', 'r') as f:
    raw_data = json.load(f)


# 2. Convert to DataFrame and flatten the nested 'parsed_data' field
df = pd.DataFrame(raw_data)
parsed_df = pd.json_normalize(df['parsed_data'])

print(parsed_df.head())

   width_ft  length_ft regional_style country_of_origin    weave_type  \
0     8.580     11.080          Heriz           Persian  Hand-Knotted   
1       NaN        NaN          Heriz           Persian  Hand-Knotted   
2     8.330     10.170          Heriz           Persian  Hand-Knotted   
3     6.417      9.583          Heriz           Persian          None   
4    10.330     12.000          Heriz           Persian          None   

        materials year_produced primary_color  extraction_confidence  \
0  [Wool, Cotton]         1930s          None                   90.0   
1  [Wool, Cotton]         1930s          None                   90.0   
2  [Wool, Cotton]         1930s          None                   90.0   
3  [Wool, Cotton]         1930s          None                   90.0   
4  [Wool, Cotton]         1930s          None                   90.0   

                             colors          condition  \
0              [Brown, Beige, Gray]  Minor Wear/Fading   
1           

In [5]:
parsed_df.columns.tolist()

['width_ft',
 'length_ft',
 'regional_style',
 'country_of_origin',
 'weave_type',
 'materials',
 'year_produced',
 'primary_color',
 'extraction_confidence',
 'colors',
 'condition',
 'patterns']

## Cleaning up the country_of_origin column

In [6]:
parsed_df['country_of_origin'].value_counts()

country_of_origin
Persian            319
Afghan              80
Chinese             49
Turkish             33
Caucasian           32
Indian              32
Kazakh              10
Moroccan             7
Afghanistan          4
Turkman              3
Native American      2
Nepal                1
Iraqi                1
Anatolia             1
Japanese             1
Bokhara              1
Morocco              1
Armenian             1
Name: count, dtype: int64

In [7]:
# Mapping specific values to standardized country names
origin_mapping = {
    'Afghan': 'Afghanistan',
    'Afghanistan': 'Afghanistan',
    'Persian': 'Iran'
}

# Apply the replacement
parsed_df['country_of_origin'] = parsed_df['country_of_origin'].replace(origin_mapping)

# Check your updated counts
parsed_df['country_of_origin'].value_counts()

country_of_origin
Iran               319
Afghanistan         84
Chinese             49
Turkish             33
Caucasian           32
Indian              32
Kazakh              10
Moroccan             7
Turkman              3
Native American      2
Nepal                1
Japanese             1
Anatolia             1
Bokhara              1
Morocco              1
Armenian             1
Iraqi                1
Name: count, dtype: int64

In [8]:
parsed_df['colors'].value_counts()

colors
[Navy Blue, Rust Red, Ivory]        55
[Multicolored]                      18
[Navy Blue]                         12
[Red, Blue, Green]                  12
[Navy Blue, Rust Red]               11
                                    ..
[Red, Blue, Black, Beige]            1
[Blue, Red, Green, Brown, Beige]     1
[Golden Yellow, Cream]               1
[Cream, Blue, Green, Red]            1
[Dark Brown]                         1
Name: count, Length: 426, dtype: int64

In [ ]:
parsed_df['primary_color'].value_counts() #outout is unexpected

primary_color
Soft colors        8
Soft Blue          6
Soft Colors        5
Blue               5
Soft colored       2
Pink               2
Light Blue         1
Classic Blue       1
light blue         1
olive              1
soft colors        1
Blue And White     1
Soft Color         1
#AC532             1
Soft Colored       1
                   1
#PS316             1
#PB306             1
Gold And Purple    1
Brown And Gold     1
#Blue And White    1
#AC277             1
Art Deco           1
Dark Blue          1
#M114              1
#F111              1
Soft               1
Soft Brown         1
Navy Blue          1
Emerald Green      1
Gold               1
blue               1
Name: count, dtype: int64

In [23]:
parsed_df['year_produced'].value_counts()

year_produced
1920s              94
1900s              86
Modern             68
1950s              59
1930s              52
1880s              41
1970s              22
Vintage            21
Antique/Unknown    13
1940s              12
null               11
1960s              10
1900                3
Traditional         2
20th century        1
1800s               1
1830s               1
1910s               1
18th Century        1
1850s               1
1880's              1
19th Century        1
Name: count, dtype: int64

In [22]:
origin_mapping = {
    'new': 'null',
    'New': 'null',
    'Modern': 'Modern',
    'Modern Design': 'Modern',
    '100% pure wool pile': 'null',
    'N/A': 'null',
    'None': 'null',
    'vintage': 'Vintage',
    'Vintage': 'Vintage',
    '1920Ss': '1920s',
    'circa 1900s': '1900s',
    'circa': 'null',
    '1920': '1920s',
    'vint': 'Vintage',
    '100% pure wool pile and cotton foundation': 'null',
    'V vintage': 'Vintage',
    'Modern design': 'Modern',
    'Antique/Unknown': 'Antique/Unknown',
    'Antique': 'Antique/Unknown',
    'antique': 'Antique/Unknown',
    'Unknown': 'Antique/Unknown',
}

parsed_df['year_produced'] = parsed_df['year_produced'].replace(origin_mapping)

# Check your updated counts
parsed_df['year_produced'].value_counts()

year_produced
1920s              94
1900s              86
Modern             68
1950s              59
1930s              52
1880s              41
1970s              22
Vintage            21
Antique/Unknown    13
1940s              12
null               11
1960s              10
1900                3
Traditional         2
20th century        1
1800s               1
1830s               1
1910s               1
18th Century        1
1850s               1
1880's              1
19th Century        1
Name: count, dtype: int64

In [33]:
parsed_df['patterns'].value_counts()

patterns
[Tribal/Geometric]                                             73
[Tribal/Geometric, Persian]                                    44
[Medallion, Geometric Stars]                                   39
[Boteh/Paisley, Medallion]                                     36
[Boteh/Paisley]                                                29
                                                               ..
[Medallion, Paisley, Geometric Stars, Tribal, Heriz, Kazak]     1
[Tribal/Geometric, Caucasian, Modern]                           1
[Birds, Paisley, Animal]                                        1
[Paisley, Medallion, Floral]                                    1
[Pictorial, Geometric, Tribal]                                  1
Name: count, Length: 187, dtype: int64